In [ ]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_rel
import scipy.stats as stats
# Load Toxic-BERT model and tokenizer
model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Read in simulated notes

In [ ]:
def read_simulated_notes(file_path):
    """
    Reads files from subfolders in the given directory into a pandas DataFrame.
    
    Args:
        file_path (str): The path to the folder containing subfolders with notes.

    Returns:
        pd.DataFrame: A DataFrame with columns 'file_name', 'note_1' to 'note_5'.
    """
    # Initialize an empty DataFrame
    data = {'file_name': []}
    for i in range(1, 6):
        data[f'note_{i}'] = []

    # Iterate over each subfolder (1-5)
    for i in range(1, 6):
        subfolder_path = os.path.join(file_path, str(i))
        if not os.path.exists(subfolder_path):
            raise ValueError(f"Subfolder {subfolder_path} does not exist.")
        
        # Read all files in the subfolder
        for file_name in os.listdir(subfolder_path):
            full_path = os.path.join(subfolder_path, file_name)
            if os.path.isfile(full_path):
                # Extract the first 16 characters of the file name
                trimmed_name = file_name[-20:-4]
                
                # Read the content of the file
                with open(full_path, 'r', encoding='utf-8') as file:
                    content = file.read()
                
                # Append the data to the dictionary
                if trimmed_name not in data['file_name']:
                    data['file_name'].append(trimmed_name)
                    for col in range(1, 6):
                        data[f'note_{col}'].append(None)
                row_idx = data['file_name'].index(trimmed_name)
                data[f'note_{i}'][row_idx] = content

    # Convert the dictionary into a DataFrame
    df = pd.DataFrame(data)
    return df

In [ ]:
# Example usage:
df = read_simulated_notes()
df.head(10)

In [ ]:
def read_reference_texts(reference_dir):
    """
    Reads text files in a directory and creates a DataFrame with two columns:
    - 'file_id': first 16 characters of each file name
    - 'content': content of the corresponding file

    Parameters:
        reference_dir (str): The directory containing reference text files.

    Returns:
        pd.DataFrame: DataFrame with file identifiers and file contents.
    """
    data = []

    # Loop through files in the directory
    for file_name in os.listdir(reference_dir):
        file_path = os.path.join(reference_dir, file_name)
        # Check if the path is a file
        if os.path.isfile(file_path):
            try:
                # Extract the first 16 characters of the file name
                file_id = file_name[:16]
                # Read the content of the file with multiple encodings fallback
                try:
                    with open(file_path, 'r', encoding='utf-8') as file:
                        content = file.read()
                except UnicodeDecodeError:
                    with open(file_path, 'r', encoding='cp1252') as file:  # Common fallback encoding
                        content = file.read()
                # Append to data list
                data.append({'file_name': file_id, 'reference_text': content})
            except Exception as e:
                print(f"Error reading {file_name}: {e}")

    # Create DataFrame
    df = pd.DataFrame(data)
    return df

In [ ]:
# Example usage
reference_dir = 
reference_df = read_reference_texts(reference_dir)
df['file_name'] = df['file_name'].str[:16]
df_with_reference = pd.merge(reference_df, df, on='file_name', how='inner')
df_with_reference.head(10)

In [ ]:
# Function to calculate toxicity
def calculate_toxicity(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    scores = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return scores[0][1].item()  # Toxicity score

# Function to calculate BLEU scores
def calculate_bleu(reference, candidate, n):
    smoothing = SmoothingFunction().method4
    weights = tuple([1.0 / n] * n) + tuple([0] * (4 - n))
    return sentence_bleu([reference.split()], candidate.split(), weights=weights, smoothing_function=smoothing)

# Function to calculate BLEU cumulative scores
def calculate_bleu_cumulative(reference, candidate):
    smoothing = SmoothingFunction().method4
    bleu1 = sentence_bleu([reference.split()], candidate.split(), weights=(1, 0, 0, 0), smoothing_function=smoothing)
    bleu2 = sentence_bleu([reference.split()], candidate.split(), weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing)
    bleu3 = sentence_bleu([reference.split()], candidate.split(), weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothing)
    bleu4 = sentence_bleu([reference.split()], candidate.split(), weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)
    return bleu1, bleu2, bleu3, bleu4

# Function to calculate ROUGE scores
def calculate_rouge(reference, candidate):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rouge3', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return scores

# Main function
def calculate_text_metrics(df, reference_col, text_cols):
    results = df.copy()
    for col in text_cols:
        rouge1_list, rouge2_list, rouge3_list, rougeL_list = [], [], [], []
        bleu1_list, bleu2_list, bleu3_list, bleu4_list = [], [], [], []
        bert_precision_list, bert_recall_list, bert_f1_list = [], [], []
        meteor_list = []
        toxicity_list = []

        for ref, text in tqdm(zip(results[reference_col], results[col]), total=len(results), desc=f"Processing {col}"):
            # ROUGE
            rouge_scores = calculate_rouge(ref, text)
            rouge1_list.append(rouge_scores['rouge1'].fmeasure)
            rouge2_list.append(rouge_scores['rouge2'].fmeasure)
            rouge3_list.append(rouge_scores['rouge3'].fmeasure)
            rougeL_list.append(rouge_scores['rougeL'].fmeasure)

            # BLEU
            bleu1_list.append(calculate_bleu(ref, text, 1))
            bleu2_list.append(calculate_bleu(ref, text, 2))
            bleu3_list.append(calculate_bleu(ref, text, 3))
            bleu4_list.append(calculate_bleu(ref, text, 4))

            # BLEU Cumulative
            bleu_cum1, bleu_cum2, bleu_cum3, bleu_cum4 = calculate_bleu_cumulative(ref, text)

            # METEOR
            try:
                meteor_list.append(meteor_score([ref.split()], text.split()))
            except:
                meteor_list.append(0.0)  # Handle empty texts

            # BERTScore
            P, R, F1 = bert_score([text], [ref], lang="en", verbose=False)
            bert_precision_list.append(P.mean().item())
            bert_recall_list.append(R.mean().item())
            bert_f1_list.append(F1.mean().item())

            # Toxicity
            toxicity_list.append(calculate_toxicity(text))

        # Append results as new columns
        results[f'{col}_rouge1'] = rouge1_list
        results[f'{col}_rouge2'] = rouge2_list
        results[f'{col}_rouge3'] = rouge3_list
        results[f'{col}_rougeL'] = rougeL_list
        results[f'{col}_bleu1'] = bleu1_list
        results[f'{col}_bleu2'] = bleu2_list
        results[f'{col}_bleu3'] = bleu3_list
        results[f'{col}_bleu4'] = bleu4_list
        results[f'{col}_meteor'] = meteor_list
        results[f'{col}_bert_precision'] = bert_precision_list
        results[f'{col}_bert_recall'] = bert_recall_list
        results[f'{col}_bert_f1'] = bert_f1_list
        results[f'{col}_toxicity'] = toxicity_list

    return results


In [ ]:
notes_columns = [col for col in df_with_reference.columns if 'note' in col]
# Run the `calculate_text_metrics` function for the notes columns
results_df = calculate_text_metrics(df_with_reference, reference_col='reference_text', text_cols=notes_columns)

results_df.head(10)

# Save dataframe

In [ ]:
# Define the metrics
metrics = ['rouge1', 'rouge2', 'rouge3', 'rougeL', 'bleu1', 'bleu2', 'bleu3', 'bleu4', 'meteor', 
           'bert_precision', 'bert_recall', 'bert_f1', 'toxicity']

# Loop through metrics to calculate averages
for metric in metrics:
    # Extract columns for the current metric across notes_1 to notes_5
    cols = [f'note_{i}_{metric}' for i in range(1, 6)]
    # Check if the columns exist in the DataFrame to avoid errors
    existing_cols = [col for col in cols if col in results_df.columns]
    if existing_cols:  # Only compute if columns are present
        # Calculate the average
        avg_col_name = f'average_{metric}'
        results_df[avg_col_name] = results_df[existing_cols].mean(axis=1)

# Display the updated DataFrame
print(results_df)

In [ ]:
# Define the save path
save_path = 

# Save the DataFrame as a CSV file
results_df.to_csv(save_path, index=False)

print(f"DataFrame saved successfully to {save_path}")

In [ ]:
# Define the file path
file_path = 

# Read the CSV file
repeated_results_df = pd.read_csv(file_path)

# Display the first few rows of the DataFrame
repeated_results_df['gpt']

In [ ]:
filtered_columns = [col for col in repeated_results_df.columns if col.startswith('gpt_') and 
                    col.endswith('_average') ]
filtered_columns.append('file_name')
# Create the sub-DataFrame
gpt_results_df = repeated_results_df[filtered_columns]
gpt_results_df.head(5)

In [ ]:
# Merge the dataframes on the 'file_name' column
merged_df = pd.merge(gpt_results_df, results_df, on='file_name', how='inner')

# Display the merged dataframe
merged_df.columns

In [ ]:
metrics = [
    ('gpt_rouge1_average', 'average_rouge1'),
    ('gpt_rouge2_average', 'average_rouge2'),
    ('gpt_rouge3_average', 'average_rouge3'),
    ('gpt_rougeL_average', 'average_rougeL'),
    ('gpt_bleu1_average', 'average_bleu1'),
    ('gpt_bleu2_average', 'average_bleu2'),
    ('gpt_bleu3_average', 'average_bleu3'),
    ('gpt_bleu4_average', 'average_bleu4'),
    ('gpt_meteor_average', 'average_meteor'),
    ('gpt_bert_precision_average', 'average_bert_precision'),
    ('gpt_bert_recall_average', 'average_bert_recall'),
    ('gpt_bert_f1_average', 'average_bert_f1'),
    ('gpt_toxicity_average', 'average_toxicity')
]

# Initialize a dictionary to store t-test results
test_results = {}

# Loop through each metric pair
for original, masked in metrics:
    # Plot histogram
    plt.figure(figsize=(10, 6))
    plt.hist(merged_df[original], bins=20, alpha=0.5, label='Original')
    plt.hist(merged_df[masked], bins=20, alpha=0.5, label='Random Mask')
    plt.xlabel('Score')
    plt.ylabel('Frequency')
    plt.title(f'Comparison of {masked.replace("average_", "")} Before and After Random Mask')
    plt.legend()
    plt.show()

    # Conduct paired t-test
    t_stat, p_value = ttest_rel(merged_df[original], merged_df[masked])
    test_results[f'{original} vs {masked}'] = {'t_stat': t_stat, 'p_value': p_value}

    # Display t-test results after each graph
    print(f"Metric: {original} vs {masked}")
    print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4f}\n")